# Feature Generation

## Scientific objective
Generate fixed-parameter Morgan fingerprints, classical descriptors, molecular-graph feature definitions, and versioned feature metadata.

## Inputs
- `data/processed/split_assignments.csv`
- `configs/model_config.yaml`

## Expected outputs
- `data/processed/morgan_features.npz`
- `data/metadata/morgan_features.json`
- `data/metadata/graph_features.json`

## Dependencies
RDKit, NumPy

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
All models use identical molecule rows and persisted split assignments. Feature preprocessing is fitted only on training folds where applicable.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
Hashed fingerprint collisions and descriptor redundancy remain possible. Graph features are 2D and do not encode conformational ensembles.

## Next notebook
[10_qsar_baselines.ipynb](./10_qsar_baselines.ipynb)

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'full', 'seed': 20260723}


In [2]:
from toxicity_screening.pipeline import generate_features
from toxicity_screening.graph_features import ATOM_FEATURE_DIM, BOND_FEATURE_DIM
from toxicity_screening.utils import atomic_write_json
index = generate_features(ROOT)
atomic_write_json({"atom_feature_dim":ATOM_FEATURE_DIM,"bond_feature_dim":BOND_FEATURE_DIM,"atom_features":CONFIGS["model_config"]["features"]["graph"]["atom_features"],"bond_features":CONFIGS["model_config"]["features"]["graph"]["bond_features"]}, ROOT / "data/metadata/graph_features.json")
archive=np.load(ROOT / "data/processed/morgan_features.npz")
assert archive["X"].shape[0] == len(index)
assert archive["X"].shape[1] == CONFIGS["model_config"]["features"]["morgan"]["n_bits"]
print(archive["X"].shape)

(25583, 2048)


In [3]:
from toxicity_screening.pipeline import generate_features
from toxicity_screening.graph_features import (
    ATOM_FEATURE_DIM,
    BOND_FEATURE_DIM,
)
from toxicity_screening.utils import atomic_write_json

index = generate_features(ROOT)

atomic_write_json(
    {
        "atom_feature_dim": ATOM_FEATURE_DIM,
        "bond_feature_dim": BOND_FEATURE_DIM,
        "atom_features": CONFIGS["model_config"]["features"]["graph"]["atom_features"],
        "bond_features": CONFIGS["model_config"]["features"]["graph"]["bond_features"],
    },
    ROOT / "data/metadata/graph_features.json",
)

print("Feature archive regenerated:", len(index))

Feature archive regenerated: 25583


In [4]:
import numpy as np

feature_path = ROOT / "data/processed/morgan_features.npz"

with np.load(feature_path, allow_pickle=False) as archive:
    X = archive["X"]
    molecule_ids = archive["molecule_id"]

print("X:", X.shape, X.dtype)
print("molecule_id:", molecule_ids.shape, molecule_ids.dtype)

assert molecule_ids.dtype.kind in {"U", "S"}
assert len(molecule_ids) == len(X)

print("Morgan archive verification passed.")

X: (25583, 2048) uint8
molecule_id: (25583,) <U27
Morgan archive verification passed.


### Completion gate
Confirm that the declared artifacts exist before continuing to `10_qsar_baselines.ipynb`.